# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MTahaR-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** My Week 1 question remains: which pages with ongoing search exposure and a recent decline in impressions should a content editor review first for a possible refresh?

Here I map that decision to a ranking task and demonstrate a historical decline-risk proxy. I used an AI assistant to help draft and execute the notebook after reading `skills/README.md`, `framing-ml-problems`, and `flyrank/flyrank-data`. The framing below explains my choices and their limits.

**Rerun:** Run all cells in order with Python 3 and pandas. The code finds the CSV in a local clone; in Colab it can read the same starter CSV from a pinned GitHub revision. References: the lane guide, data dictionary, and my Week 1 notebook.

## 1. My lane as an ML task (type)

My primary task is **ranking/scoring**: order eligible pages within each client so an editor can spend limited review time on the most relevant candidates. A possible implementation is binary classification followed by ranking pages by estimated decline risk. Classification is a means to produce the queue; accuracy across all pages is not my main success measure.

**The ML loop:** define the review decision → choose the page/decision-window grain → collect inputs available before the outcome window → define an observed later-window target → build a fixed-rule baseline → compare a learned score on held-out data → return a ranked queue with reasons → let an editor investigate and choose an action → monitor later outcomes.

The action remains a **human review for a possible refresh**. An editor checks intent, content, related pages, and context before deciding to update, investigate further, or monitor. A wrong recommendation wastes review time or encourages an unnecessary edit; a missed candidate may leave an emerging problem unattended.

In [1]:
from pathlib import Path
import pandas as pd

relative_path = Path("data/raw/content_refresh_anonymized.csv")
data_path = next(
    (parent / relative_path for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / relative_path).is_file()),
    None,
)
starter_revision = "224d6365f72f39b6ce899376f8fe7b6d3a34265e"
starter_url = (
    "https://raw.githubusercontent.com/MTahaR-dev/flyrank-ml-internship/"
    f"{starter_revision}/{relative_path.as_posix()}"
)
raw = pd.read_csv(data_path if data_path is not None else starter_url)
assert raw["content_id"].notna().all() and raw["content_id"].is_unique
print(f"Loaded {len(raw):,} starter pages; content IDs are unique.")

Loaded 30,000 starter pages; content IDs are unique.


## 2. Target or proxy

For this historical sketch, the decision point **t0** is the start of the most recent 30-day window in the snapshot. Inputs come from the preceding 30 days; the outcome comes from the following 30 days, which are already observed in this retrospective dataset.

**Provisional target:** `future_decline_30d_proxy = 1` when impressions in the later 30 days are **more than 20% lower** than in the preceding 30 days; otherwise it is `0`. The 20% cutoff turns an observed outcome into an operational label. It does not establish the cause, persistence, business impact, or benefit of a refresh. “Future” means later than this historical decision point, not later than the dataset export.

**Eligible slice:** pages with at least **100 impressions in the preceding 30 days**. This is a provisional monthly exposure floor, motivated by Week 1's 300-impression/90-day screen. It is not the same filter. I do not prefilter to declining pages, because the sketch needs both positive and negative outcomes and eligibility must be known at t0. Update-age metadata measured at export is also not used as a historical t0 feature here.

This target agrees with the starter `down` bucket, so reproducing that bucket from `trend_pct` would be circular. The label is calculated from the actual two-window counts, and the later-window count is retained only as target evidence. I exclude `trend_pct`, `trend_direction`, all last-30-day metrics, and the 90-day totals from the model inputs because they contain outcome-window information.

For the later capstone, I will construct explicit dated windows in the warehouse, investigate persistence and competing explanations, and test on untouched later periods. This notebook does not yet predict continued decline after today's snapshot or determine which edit would work.

In [2]:
MIN_PREV30_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.20

required_counts = ["impressions_prev_30d", "impressions_last_30d"]
eligible = (
    raw[required_counts].notna().all(axis=1)
    & raw["impressions_prev_30d"].ge(MIN_PREV30_IMPRESSIONS)
)
lane = raw.loc[eligible].copy()
lane["future_decline_30d_proxy"] = (
    lane["impressions_last_30d"]
    .lt((1 - DECLINE_THRESHOLD) * lane["impressions_prev_30d"])
    .astype("int64")
)
assert lane["content_id"].is_unique
assert set(lane["future_decline_30d_proxy"].unique()) == {0, 1}

print(f"Eligible slice: {len(lane):,} pages across {lane['client_id'].nunique()} clients.")
print(f"Observed later-window decline proxy: {lane['future_decline_30d_proxy'].sum():,} "
      f"pages ({lane['future_decline_30d_proxy'].mean():.2%}).")

# Aggregate target sketch: real observed labels, not invented examples.
target_counts = (
    lane["future_decline_30d_proxy"].value_counts()
    .reindex([0, 1], fill_value=0).rename_axis("future_decline_30d_proxy")
    .reset_index(name="pages")
)
target_counts

Eligible slice: 18,010 pages across 30 clients.
Observed later-window decline proxy: 11,090 pages (61.58%).


,future_decline_30d_proxy,pages
0,0,6920
1,1,11090


## 3. Success metric

My primary metric is **mean per-client Precision@20**. For each client, it is the number of proxy-positive pages in the first 20 recommendations divided by 20. I then average those values equally across clients so one large client's inventory does not dominate the result. Twenty reviews per client per cycle is an assumed capacity to revisit, not a known staffing fact. Clients with fewer than 20 eligible pages are excluded from this exact-K metric and will need a separate small-queue policy.

**What would count as good:** on the same held-out evaluation, a learned ranking should improve mean Precision@20 by at least **0.05** over the stronger of the fixed-rule baseline and the expected random ranking within each client. That is about one extra relevant candidate per 20 reviews on average. The five-percentage-point margin is a provisional practical goal; I will also examine uncertainty across clients and time before claiming an improvement.

**Baseline:** rank by preceding-window impressions, highest first, with pseudonymized page ID used only to break ties reproducibly. This favors visible pages and uses no later-window data. A uniformly random within-client ranking has expected Precision@20 equal to that client's positive rate.

The code below computes both reference values on this historical slice to make the metric concrete. These are **descriptive baseline results**, not held-out model validation. No model has been fitted or compared. I will use whole-client holdout for a starter comparison and dated, non-overlapping windows for warehouse validation; the final test data must remain untouched during development.

In [3]:
K = 20
client_sizes = lane.groupby("client_id")["content_id"].transform("size")
evaluation_rows = lane.loc[client_sizes.ge(K)].copy()
assert not evaluation_rows.empty

baseline_ranked = evaluation_rows.sort_values(
    ["client_id", "impressions_prev_30d", "content_id"],
    ascending=[True, False, True],
)
top_k = baseline_ranked.groupby("client_id", sort=False).head(K)
assert top_k.groupby("client_id").size().eq(K).all()

baseline_p20_by_client = top_k.groupby("client_id")["future_decline_30d_proxy"].mean()
random_expectation_by_client = (
    evaluation_rows.groupby("client_id")["future_decline_30d_proxy"].mean()
)
baseline_p20 = baseline_p20_by_client.mean()
random_expected_p20 = random_expectation_by_client.mean()

print(f"Exact-{K} metric population: {len(evaluation_rows):,} pages across "
      f"{evaluation_rows['client_id'].nunique()} clients.")
print(f"Clients with fewer than {K} eligible pages excluded: "
      f"{lane['client_id'].nunique() - evaluation_rows['client_id'].nunique()}.")
metric_summary = pd.DataFrame({
    "reference": ["Prior-impressions fixed rule", "Expected within-client random ranking"],
    "mean_per_client_precision_at_20": [baseline_p20, random_expected_p20],
})
metric_summary.round(4)

Exact-20 metric population: 17,953 pages across 22 clients.
Clients with fewer than 20 eligible pages excluded: 8.


,reference,mean_per_client_precision_at_20
0,Prior-impressions fixed rule,0.5273
1,Expected within-client random ranking,0.6125


**Interpretation:** The exposure-only rule has lower mean Precision@20 than the expected random ranking on this slice. Exposure alone does not prioritize this decline proxy well. That motivates testing other pre-decision signals, but does not prove ML will succeed. The reference numbers are not a forecast of performance on future periods, and the metric does not measure recovered traffic or edit effectiveness.

## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized page at one historical decision window**, belonging to one pseudonymized client. The starter contains one such example per page, not one row per day, click, or client. A future warehouse table would have one row per page/decision date, with repeated examples handled carefully during splitting.

The dataframe below shows actual eligible rows and their target evidence. The first three numeric columns are a small provisional input set from before t0. `impressions_last_30d` is outcome evidence, not an input. The separate `X` and `y` objects make that boundary explicit. Page and client IDs support grouping, joins, and split checks; they are not features.

I will audit measurement availability before training: missing values will not be silently replaced with zeros, and GA4 availability must be checked when I move to warehouse data.

In [4]:
FEATURE_COLUMNS = [
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]
TARGET_COLUMN = "future_decline_30d_proxy"
ID_COLUMNS = ["content_id", "client_id"]

unit_of_analysis = lane[
    ID_COLUMNS + FEATURE_COLUMNS + ["impressions_last_30d", TARGET_COLUMN]
].copy()
X = unit_of_analysis[FEATURE_COLUMNS].copy()
y = unit_of_analysis[TARGET_COLUMN].copy()
groups = unit_of_analysis["client_id"].copy()

assert unit_of_analysis["content_id"].is_unique
assert X.index.equals(y.index) and y.index.equals(groups.index)
assert not set(ID_COLUMNS + [TARGET_COLUMN, "impressions_last_30d"]).intersection(X.columns)
assert all(column.endswith("_prev_30d") for column in X.columns)

print(f"Page-level dataframe: {unit_of_analysis.shape[0]:,} rows, "
      f"{unit_of_analysis.shape[1]} columns.")
print(f"X: {X.shape}; y: {y.shape}. No IDs or outcome-window fields in X.")
print("Rows below are real starter records with pseudonymized IDs.")
unit_of_analysis.head(8)

Page-level dataframe: 18,010 rows, 7 columns.
X: (18010, 3); y: (18010,). No IDs or outcome-window fields in X.
Rows below are real starter records with pseudonymized IDs.


,content_id,client_id,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,impressions_last_30d,future_decline_30d_proxy
0,content_304f48230142,client_f369cb89fc,987,13,9,578,1
1,content_a1fb4e703a9e,client_4e07408562,5915,1,2,2501,1
2,content_9aa793d4d895,client_7f2253d7e2,6089,3,3,2382,1
3,content_331d6c4de07b,client_19581e27de,4206,17,26,3626,0
4,content_d99b7a2d90ca,client_3fdba35f04,6452,2,9,4211,1
5,content_d4084a4bc775,client_f369cb89fc,1009,1,1,617,1
7,content_a63219c6e95a,client_19581e27de,632,0,4,636,0
8,content_5e6c160719bc,client_6208ef0f77,13828,8,14,5696,1


## 5. Why ML beats a fixed rule here

**I have not shown that ML beats a fixed rule yet.** My hypothesis is that a learned score may combine prior exposure, clicks, sessions, and later carefully constructed historical trend/context features more effectively than one fixed exposure cutoff or ordering. Relationships may differ across clients and traffic levels, so interactions could matter. That is an empirical question, not permission to assume a complex model is better.

The measured baseline result gives me a concrete starting point to challenge. I will first audit the inputs and labels, then compare a simple learned ranking with the fixed rule on held-out data using the metric defined above. If the model fails to improve it reliably, I will keep the transparent rule or analysis and explain the result. The goal is a better allocation of review time, not a trained model for its own sake.

The queue should show measured reasons and uncertainty. A high estimated decline risk suggests a page deserves investigation; it does not tell the editor which change will help. Seasonality, consolidation, changes in search demand, and noise remain possible explanations. I can make observational and decision-support claims, but cannot claim that refreshing a page causes recovery or that I have discovered Google's ranking factors.

## 6. Self-check

- [x] Named Lane 2 as ranking/scoring, with classification as a possible implementation.
- [x] Defined a binary target from actual later-window measurements and explained its proxy status.
- [x] Separated pre-decision inputs from label-window evidence and excluded IDs from features.
- [x] Named Precision@20, its denominator, the client aggregation, and a provisional improvement goal.
- [x] Computed a fixed-rule reference and matched random expectation without claiming model validation.
- [x] Loaded the lane slice and displayed an actual page-level dataframe and target counts.
- [x] Tied the queue to editorial review and explained the cost of false positives and missed candidates.
- [x] Explained why ML might help and that beating a rule still needs evidence.
- [x] Executed all code cells in order with no errors and retained their outputs.
- [x] Used public-safe pseudonymized records and careful observational language.

**Submission:** Commit and push this executed notebook at `work/notebooks/w02_ml_task_framing.ipynb`, check that its outputs appear on GitHub, and submit the repository URL on the assignment card.